In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from bingX import BingX
import time

    
engulfing_body_condition = 2
threesoldiers_body_condition = [0.5,0.5]


secret_key = "WqP5txrWtmbGcMND5GjFiBZICth5nZFIXYKEPqKitRl7xOHVGOFzNo8yNCLQUmK4guPTNgV4nwKLT0aD5iYIg"
api_key = "SbyekABHMR0SyrIQKTqU14sExh5m40sV5a6jaBenI26u5SDm8z4gGzQCLamCp6Ww2uJyVjyhJQESk4Gt5jA"


## functions

In [2]:
from enum import Enum

class Strategy(Enum):
    THREESOLDIERS = 4
    ENGULFING = 3
    
class Interval(Enum):
    HOURLY_ONEHOUR = '1h'
    HOURLY_FOURHOUR = '4h'
    DAILY_ONEDAY = '1d'
    MINUTELY_ONEMINUTE = '1m'
    
class Audit(Enum):
    WEEKLY = 1
    MONTHLY = 2

def updateTotalBalance(i):
    global usdt_balance, bitcoin_balance, total_balance
    total_balance = usdt_balance + bitcoin_balance * cdf.loc[i]["close"]

def buyBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global buyDuplicateChecker,sellDuplicateChecker, learning_rate, buy_percentage_current
    global buy_percentage_init, sell_percentage_current, sell_percentage_init
   
    buyDuplicateChecker += 1
    sellDuplicateChecker = 0
    if strategy.name == Strategy.ENGULFING.name:
        if buyDuplicateChecker != 3:
            return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[92m \033[1m{}\033[00m" .format(f"buy opportunity!\n"))        

    if usdt_balance > 0:
        buyPositionsNumber += 1
        
        print(f'current buy percentage: {buy_percentage_current}')
        amountToBuy_usdt = total_balance * buy_percentage_current / 100
        buy_percentage_current *= learning_rate
        sell_percentage_current = sell_percentage_init
        
        if amountToBuy_usdt > usdt_balance:
            amountToBuy_usdt = usdt_balance
            
        addedBitcoinAmount = np.longdouble(amountToBuy_usdt / cdf.loc[i]["close"])
        usdt_balance -= amountToBuy_usdt
        bitcoin_balance += addedBitcoinAmount
        updateTotalBalance(i)
        print(f'\tbought {amountToBuy_usdt} $ BTC ({addedBitcoinAmount} BTCs) in {time.ctime(cdf.loc[i]["time"] // 1e3)}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')
        
    else:
        print(f"\tthere isn't any usdt to buy BTC with it!")
        print(f'\tcurrent BTC balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')

def sellBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global sellDuplicateChecker,buyDuplicateChecker, buy_percentage_current, buy_percentage_init
    global sell_percentage_current, sell_percentage_init, learning_rate
    
    sellDuplicateChecker += 1
    buyDuplicateChecker = 0
    
    if strategy.name == Strategy.ENGULFING.name:
        if sellDuplicateChecker != 3:
            return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[91m \033[1m{}\033[00m" .format(f"sell opportunity!\n"))        
    if bitcoin_balance > 0:
        sellPositionsNumber += 1
        
        currentValueOfAllBitcoins = bitcoin_balance * cdf.loc[i]["close"]
        
        print(f'current sell percentage: {sell_percentage_current}')
        amountToSell_usdt = total_balance * sell_percentage_current / 100
        sell_percentage_current *= learning_rate
        buy_percentage_current = buy_percentage_init
        
        if amountToSell_usdt > currentValueOfAllBitcoins:
            amountToSell_usdt = currentValueOfAllBitcoins
        
        amountToSell_bitcoin = amountToSell_usdt / cdf.loc[i]["close"]
        usdt_balance += amountToSell_usdt
        bitcoin_balance -= amountToSell_bitcoin
        updateTotalBalance(i)
        print(f'\tsold {amountToSell_usdt} $ of BTC ({amountToSell_usdt} $) in {time.ctime(cdf.loc[i]["time"] // 1e3)}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["close"]} $)')
        
    else:
        print(f"\tthere isn't any bitcoin to sell!")
        print(f'\tcurrent USDT balance: {usdt_balance} $')

def checkForPosition(i,strategy):
    if strategy == Strategy.THREESOLDIERS:
        if i >= len(cdf) - 2:
            return
        suitableForBuy = False
        suitableForSell = False
        print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 1.0:
                if cdf.loc[2 + i]['isGreen'] == 1.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 0.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForSell = True
                
    if strategy == Strategy.ENGULFING:
        if i >= len(cdf) - 1:
            return
        suitableForBuy = False
        suitableForSell = False
        print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForSell = True
                

        if suitableForBuy:
    #         print(f'{i} is suitable for buy!')
            buyBTC(i)

        if suitableForSell:
    #         print(f"{i} is suitable for sell!")
            sellBTC(i)

    #     if not (suitableForBuy or suitableForSell):
    #         print(f'{i} is not suitable for buy nor sell!')

def timeout(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber, counter
    updateTotalBalance(i)
    print("\033[91m \033[1m{}\033[00m" .format(f"\n\t\tCOUNTER REACHER IT'S MAXIMUM!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\topened {buyPositionsNumber} buy\
positions and {sellPositionsNumber} sell positions!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent usdt balance: {usdt_balance}\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]['Close']} $)\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent total balance: {total_balance}\n"))
    counter = 0
    buyPositionsNumber = 0
    sellPositionsNumber = 0
    return total_balance

def calculatePeriod():
    global audit, interval, period
    if audit == Audit.MONTHLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 30
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 30
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 30
            
    if audit == Audit.WEEKLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 7
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 7
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 7



## end functions

In [3]:
pair = "BTC-USDT"
interval = Interval.MINUTELY_ONEMINUTE
strategy = Strategy.ENGULFING
audit = Audit.WEEKLY

communication_max_tries = 5

buy_percentage_init = 10
sell_percentage_init = 13
learning_rate = 1.3

usdt_balance = 0
bitcoin_balance: np.longdouble = 0
total_balance = 0 # usdt_balance + bitcoin_balance * bitcoin_price
buyPositionsNumber = 0
sellPositionsNumber = 0
buyDuplicateChecker = 0
sellDuplicateChecker = 0
period = 0
buy_percentage_current = buy_percentage_init
sell_percentage_current = sell_percentage_init
usdt_balance_init = usdt_balance


print(f'chosen strategy: {strategy.name}')

chosen strategy: ENGULFING


In [4]:
if interval.value == '1h':
    timeOffset = 60 * 60 * 1000
if interval.value == '4h':
    timeOffset = 4 * 60 * 60 * 1000
if interval.value == '1d':
    timeOffset = 24 * 60 * 60 * 1000
if interval.value == '1m':
    timeOffset = 60 * 1000
    
start_time = round(time.time() * 1e3) - timeOffset * (strategy.value)
end_time = round(time.time() * 1e3)
print(f'difference(ms): {end_time - start_time}')
print(f'difference(hour): {(end_time - start_time) / 1e3 / 60 / 60}')

difference(ms): 180000
difference(hour): 0.05


In [5]:
bingx_client = BingX(api_key=api_key, secret_key=secret_key)

In [6]:
bingx_client.perpetual_v2.

{'bids': [['69080.03', '2.53463'],
  ['69079.99', '6.32070'],
  ['69079.96', '2.61346'],
  ['69079.95', '3.12566'],
  ['69079.93', '3.08689'],
  ['69079.90', '6.22148'],
  ['69079.87', '2.84127'],
  ['69079.86', '3.08689'],
  ['69079.83', '3.60801'],
  ['69079.81', '2.78718'],
  ['69079.80', '3.26105'],
  ['69079.78', '2.62859'],
  ['69079.76', '3.30523'],
  ['69079.75', '3.44901'],
  ['69079.73', '2.95237'],
  ['69079.71', '2.48704'],
  ['69079.68', '16.83495'],
  ['69079.67', '2.62859'],
  ['69079.65', '3.44901'],
  ['69079.61', '19.31021']],
 'asks': [['69081.76', '1.06516'],
  ['69081.72', '1.31709'],
  ['69081.71', '4.08070'],
  ['69081.70', '1.03736'],
  ['69081.69', '23.80364'],
  ['69081.68', '14.48545'],
  ['69081.66', '8.37760'],
  ['69081.63', '5.18828'],
  ['69081.61', '2.97073'],
  ['69081.58', '3.89480'],
  ['69081.57', '1.36760'],
  ['69081.54', '1.20922'],
  ['69081.53', '1.19309'],
  ['69081.52', '1.16588'],
  ['69081.50', '2.48121'],
  ['69081.49', '1.42364'],
  ['690

In [172]:
candleData = bingx_client.perpetual_v2.market.get_k_line_data(pair,interval.value,start_time,end_time)
candleData # last index is the last candle and so forth

[{'open': '64591.3',
  'close': '64599.3',
  'high': '64621.9',
  'low': '64570.8',
  'volume': '26.14',
  'time': 1711199220000},
 {'open': '64598.6',
  'close': '64571.0',
  'high': '64633.5',
  'low': '64565.4',
  'volume': '53.55',
  'time': 1711199280000}]

In [173]:
import warnings
warnings.filterwarnings('ignore')

df = pd.DataFrame(candleData)
cdf = df.iloc[::-1]

cdf['open'] = pd.to_numeric(cdf['open'])
cdf['high'] = pd.to_numeric(cdf['high'])
cdf['low'] = pd.to_numeric(cdf['low'])
cdf['close'] = pd.to_numeric(cdf['close'])

cdf['isGreen'] = cdf["close"]
cdf['bodyLen'] = abs(cdf["close"] - cdf["open"])

cdf = cdf.reset_index()
del cdf['index']

for i in range(len(cdf)):
    tmp = cdf.loc[i,cdf.columns.values]
    if tmp['open'] <= tmp['close']:
        tmp['isGreen'] = 1
    else:
        tmp['isGreen'] = 0

    cdf.loc[i] = tmp.values

cdf

,open,close,high,low,volume,time,isGreen,bodyLen
0,64598.6,64571.0,64633.5,64565.4,53.55,1711199280000,0.0,27.6
1,64591.3,64599.3,64621.9,64570.8,26.14,1711199220000,1.0,8.0


In [174]:
cdf.describe()

,open,close,high,low,time,isGreen,bodyLen
count,2.00000,2.000000,2.000000,2.000000,2.000000e+00,2.000000,2.000000
mean,64594.95000,64585.150000,64627.700000,64568.100000,1.711199e+12,0.500000,17.800000
std,5.16188,20.011122,8.202439,3.818377,4.242641e+04,0.707107,13.859293
min,64591.30000,64571.000000,64621.900000,64565.400000,1.711199e+12,0.000000,8.000000
25%,64593.12500,64578.075000,64624.800000,64566.750000,1.711199e+12,0.250000,12.900000
50%,64594.95000,64585.150000,64627.700000,64568.100000,1.711199e+12,0.500000,17.800000
75%,64596.77500,64592.225000,64630.600000,64569.450000,1.711199e+12,0.750000,22.700000
max,64598.60000,64599.300000,64633.500000,64570.800000,1.711199e+12,1.000000,27.600000


## dataset is ready

In [175]:
res = bingx_client.spot.trade.get_assets()
res

{'balances': [{'asset': 'BTC', 'free': '0.00041958', 'locked': '0'},
  {'asset': 'USDT', 'free': '62.17024165212848', 'locked': '20.718'},
  {'asset': 'VST', 'free': '0', 'locked': '0'},
  {'asset': 'XRP', 'free': '0', 'locked': '0'}]}

In [176]:
for record in res['balances']:
    if record['asset'] == 'USDT':
        usdt_balance = float(record['free'])
    if record['asset'] == 'BTC':
        bitcoin_balance = float(record['free'])

print(f'usdt balance: {usdt_balance}\n')
print(f'bitcoin balance: {bitcoin_balance}')

usdt balance: 62.17024165212848

bitcoin balance: 0.00041958


In [177]:
calculatePeriod()

In [178]:
period

0

In [179]:
totals = []
counter = 0
total_counter = 0

calculatePeriod()
for i in range(len(cdf) -1, -1, -1):
    checkForPosition(i,strategy)
    counter += 1
    if counter == period:
        totals.append(timeout(i))

i = 0:
